md

In [ ]:
# Registered constants, read from the pre-registration at build time.
SEEDS_HERE = [1401, 1402, 1403]
STOP_SPREAD_HERE = 0.15
print('registered seeds:', SEEDS_HERE, '| stopping threshold:', STOP_SPREAD_HERE)

In [ ]:
import glob, json, os, shutil, subprocess, sys

work = '/kaggle/working/kalia'
if not os.path.exists(work):
    # Prefer the kalia-code-dev dataset. Any train.py in /kaggle/input also
    # matches a stale repo bundled inside old kernel outputs (a Sep-23
    # snapshot), which silently lacked tools/forgetting_probe.py -- the
    # same silent-wrong-source defect class as I17. Assert the source.
    hits = sorted(glob.glob('/kaggle/input/**/train.py', recursive=True))
    preferred = [p for p in hits if '/kalia-code-dev/' in p]
    assert preferred, ('kalia-code-dev is not attached or lacks train.py; '
                       'other train.py found (do NOT trust them): ' + str(hits))
    shutil.copytree(os.path.dirname(preferred[0]), work)
os.chdir(work)
for req in ('train.py', 'tools/forgetting_probe.py', 'configs/kalia-v020.yaml',
            'mixture.py', 'make_replay_shard.py'):
    assert os.path.exists(req), f'{req} missing from the copied code snapshot'
data = sorted(glob.glob('/kaggle/input/**/train.bin', recursive=True))
assert data, 'train.bin not found - attach a prep output as a kernel source'
DATA = os.path.dirname(data[0])
print('code:', work)
print('data:', DATA)
import torch
print('gpu:', torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else '')
print('adaptive_every supported:', 'adaptive_every' in open('train.py').read())

In [ ]:
arms = ['v030-replay-00-s1401', 'v030-replay-10-s1401', 'v030-replay-40-s1401']
for a in arms:
    assert os.path.exists(f'configs/{a}.yaml'), f'missing configs/{a}.yaml'
    print(a)
print('\nRun Step 2 first: the LR schedule must be fixed before a ratio is')
print('meaningful, and Step 1 supplies the bar.')
ckpt_hits = sorted(glob.glob('/kaggle/input/**/ckpt.pt', recursive=True))
RESUME_CKPT = ckpt_hits[0] if ckpt_hits else 'kalia-lm/kalia-v020'
print('resume checkpoint:', RESUME_CKPT)


In [ ]:
import pathlib, re
prev2 = pathlib.Path('/kaggle/working/step2_rewarm.json')
if not prev2.exists():
    prev2 = pathlib.Path('/kaggle/input/step2_rewarm.json')
assert prev2.exists(), (
    'Step 2 output not found. Run kalia-v030-rewarm first: the replay\n'
    '    arms must use the schedule Step 2 validated, not a hardcoded one.')
step2 = json.load(open(prev2))
rewarm = bool(step2['passes'])
print(f"Step 2 verdict: {'TRANSFERS -> re-warm ON' if rewarm else 'NO TRANSFER -> re-warm OFF'}")
for a in arms:
    p = f'configs/{a}.yaml'
    txt = open(p).read()
    txt2, n = re.subn(r'(?m)^  rewarm:.*$', f'  rewarm: {str(rewarm)}', txt)
    assert n == 1, f'{p}: expected exactly one rewarm key'
    open(p, 'w').write(txt2)
    print(f'  {a}: rewarm: {str(rewarm)}')
print('Caution: these configs live in the kernel\'s copy of the repo. '
      'The committed configs are unchanged; the run log records the choice.')

In [ ]:
print(subprocess.run([sys.executable, 'ablate.py', '--arms', ','.join(arms),
                       '--steps', '500', '--data-dir', DATA,
                       '--resume-from', RESUME_CKPT,
                       '--out-root', '/kaggle/working/out'],
                      capture_output=True, text=True).stdout[-6000:])

In [ ]:
import pandas as pd
BAR = json.load(open('/kaggle/working/step1_spread.json'))['bar']
losses = {}
for a in arms:
    df = pd.read_csv(f'/kaggle/working/out/{a}/val_log.csv')
    print(f'{a:26s} ' + '  '.join(f"{int(r.step)}={r.val_loss:.4f}" for r in df.itertuples()))
    losses[a] = float(df.iloc[-1].val_loss)
base = losses['v030-replay-00-s1401']
print(f'\nbaseline 0%: {base:.4f} | bar {BAR:.4f}')
best_arm, best = None, None
for a in arms[1:]:
    d = losses[a] - base
    mark = 'BEATS' if d <= -BAR else 'within noise'
    print(f'  {a:26s} {losses[a]:.4f}  delta {d:+.4f}  {mark}')
    if best is None or losses[a] < best:
        best, best_arm = losses[a], a
matters = best is not None and (base - best) > BAR
print(f'\nratio matters here: {"YES" if matters else "NO"} '
      f'(best {best_arm})')
if not matters:
    print('  All three within the measured spread. That is a result: at 58M the')
    print('  replay ratio is not the delicate knob the folklore suggests, which')
    print('  is what the 405M paper reports too -- at a scale nobody extrapolated to.')